# 2025-10-24: BMMC and PBMC Frequency Analysis
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology

**Method:**

1. Flu Analysis: Unpaired Wilcoxon rank sum test, followed by linear regression adjusting for sex.
2. Treatment Timepoint Analysis: Paired Wilcoxon rank sum test.
3.  Healthy vs Treatment Analysis: Unpaired Wilcoxon rank sum test.
   - FDR Adjustment: Conducted across cell types per hypothesis using the Benjamini-Hochberg method.

In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(purrr)
  library(ggplot2)
  library(ggpubr)
  library(rstatix)
  library(forcats)
  library(tidyr)
  library(ggrepel)
  library(rlang)
})
options(repr.plot.width = 11, repr.plot.height = 6)

## Code for Frequency Analysis
This R function performs paired Wilcoxon signed-rank tests on cell type frequency data between two time points/conditions. Compares CLR-transformed frequencies for each cell type across paired samples (e.g., pre/post treatment), with FDR correction for multiple testing.

In [2]:
# Paired Wilcoxin Test
test_wilcox_paired <- function(freq_df, g1, g2,
                               freq_data = 'cell_type_clr',
                               analysis_label,
                               label_col = 'aifi_plot_l3') {
  freq_df %>%
    filter(label.visitDetails %in% c(g1, g2)) %>% # keep specified visits
    distinct(subject.subjectGuid, .data[[label_col]], label.visitDetails, .data[[freq_data]]) %>% # unique rows
    pivot_wider(
      id_cols = c(subject.subjectGuid, .data[[label_col]]),
      names_from = label.visitDetails,
      values_from = any_of(freq_data)
    ) %>%
    drop_na(all_of(c(g1, g2))) %>% # complete pairs only
    group_by(.data[[label_col]]) %>%
    reframe({
      wt <- wilcox.test(.data[[g1]], .data[[g2]], paired = TRUE, exact = TRUE) # run paired Wilcoxon test
      tibble(
        n = n(),
        statistic = unname(wt$statistic),
        p = wt$p.value
      )
    }) %>%
    mutate(
      group1 = g1,
      group2 = g2,
      analysis_label = analysis_label,
      .before = 1
    ) %>%
    arrange(.data[[label_col]]) %>%
    mutate(p.adj = p.adjust(p, method = 'fdr')) # FDR correction on the condition set
}

In [3]:
# Unpaired Wilcoxin Test
test_wilcox_unpaired <- function(freq_df, g1, g2,
                                 freq_data = 'cell_type_clr',
                                 analysis_label,
                                 label_col = 'aifi_plot_l3') {
  freq_df %>%
    filter(label.visitDetails %in% c(g1, g2)) %>% # keep specified visits
    distinct(subject.subjectGuid, .data[[label_col]], label.visitDetails, .data[[freq_data]]) %>% # unique rows
    group_by(.data[[label_col]]) %>%
    reframe({
      x <- .data[[freq_data]][label.visitDetails == g1]
      y <- .data[[freq_data]][label.visitDetails == g2]
      wt <- wilcox.test(x, y, paired = FALSE, exact = TRUE)
      tibble(
        n1 = length(x),
        n2 = length(y),
        statistic = unname(wt$statistic),
        p = wt$p.value
      )
    }) %>%
    mutate(
      group1 = g1,
      group2 = g2,
      analysis_label = analysis_label,
      .before = 1
    ) %>%
    arrange(.data[[label_col]]) %>%
    mutate(p.adj = p.adjust(p, method = 'fdr')) # FDR correction on the condition set
}

In [4]:
# Calculate effect size
calc_effect_size <- function(freq_df, g1, g2, freq_data, label_col, paired = FALSE) {
  freq_df <- freq_df %>%
    filter(.data[['label.visitDetails']] %in% c(g1, g2)) %>%
    distinct(
      .data[['subject.subjectGuid']],
      .data[[label_col]],
      .data[['label.visitDetails']],
      .data[[freq_data]]
    )
  if (paired) { ## for paired wilcoxon test
    freq_df %>%
      pivot_wider(
        id_cols = c(.data[['subject.subjectGuid']], .data[[label_col]]),
        names_from = .data[['label.visitDetails']],
        values_from = any_of(freq_data)
      ) %>%
      drop_na(all_of(c(g1, g2))) %>%
      group_by(.data[[label_col]]) %>%
      summarise(
        diff = median(.data[[g2]] - .data[[g1]], na.rm = TRUE),
        .groups = 'drop'
      )
  } else { ## for unpaired wilcoxon test
    freq_df %>%
      group_by(.data[[label_col]]) %>%
      summarise(
        diff = stats::median(.data[[freq_data]][label.visitDetails == g2], na.rm = TRUE) -
               stats::median(.data[[freq_data]][label.visitDetails == g1], na.rm = TRUE),
        .groups = 'drop'
      )
  }
}

In [5]:
# Plot frequency volcano
plot_freq_volcano <- function(freq_df, deg_res,
                              freq_data = 'cell_type_clr',
                              label_col = 'aifi_plot_l3',
                              g1_col = 'group1',
                              g2_col = 'group2',
                              alpha = 0.05,
                              effect_size = 0,
                              title = NULL,
                              paired = FALSE,
                              col_g1 = 'red4',
                              col_g2 = 'green4') {
  stopifnot(all(c(
    g1_col, g2_col, label_col,
    'p', 'p.adj'
  ) %in% names(deg_res)))

  g1 <- unique(deg_res[[g1_col]])[1]
  g2 <- unique(deg_res[[g2_col]])[1]

  out_prefix <- paste0(
    tolower(gsub('\\s+', '_', unique(freq_df$tissue))), '_',
    tolower(gsub('\\s+', '_', g1)), '_vs_', tolower(gsub('\\s+', '_', g2))
  )

  eff <- calc_effect_size(freq_df, g1, g2, freq_data, label_col, paired = paired)

  res_sig <- deg_res %>%
    select(all_of(c(
      g1_col, g2_col, label_col,
      'p', 'p.adj'
    ))) %>%
    left_join(eff, by = setNames(label_col, label_col)) %>%
    mutate(
      padj = p.adj,
      neglog = -log10(pmax(padj, .Machine$double.eps)),
      sig = padj <= alpha & abs(diff) > effect_size,
      dir = case_when(
        sig & diff > 0 ~ paste('Up in', g2),
        sig & diff < 0 ~ paste('Up in', g1),
        TRUE ~ 'Not sig'
      )
    )

  col_map <- c(
    setNames(col_g1, paste('Up in', g1)),
    setNames(col_g2, paste('Up in', g2)),
    setNames('grey70', 'Not sig')
  )

  p <- ggplot2::ggplot(res_sig, ggplot2::aes(x = diff, y = neglog, color = dir)) +
    ggplot2::geom_point(size = 3, alpha = 0.8) +
    ggrepel::geom_text_repel(
      data = res_sig %>% filter(sig),
      ggplot2::aes(label = .data[[label_col]]),
      size = 4, max.overlaps = Inf, min.segment.length = 0
    ) +
    ggplot2::geom_hline(yintercept = -log10(alpha), linetype = 'dashed') +
    ggplot2::geom_vline(xintercept = c(-effect_size, effect_size), linetype = 'dashed') +
    ggplot2::labs(
      title = title %||% paste0(g1, ' vs ', g2),
      subtitle = paste0(unique(freq_df$tissue), ' - ', unique(deg_res$analysis_label)),
      x = if (paired) paste0('Median paired difference (', g2, ' − ', g1, ')') else paste0('Median difference (', g2, ' − ', g1, ')'),
      y = '-log10(FDR)',
      color = NULL
    ) +
    ggplot2::scale_color_manual(values = col_map) +
    ggplot2::theme_bw(base_size = 14)

  # CSV into results/freq_results
  write.csv(res_sig, file.path('../../../data/rna/pseudobulk/results', 'freq_results', paste0(out_prefix, '_results.csv')))

  # Plot into results/freq_plots
  ggsave(file.path('../../../data/rna/pseudobulk/results', 'freq_plots', paste0(out_prefix, '_plot.png')),
    plot = p, width = 7, height = 5, dpi = 300
  )
  return(p) # uncomment for inline plots
}

In [6]:
dir.create('../../../data/rna/pseudobulk/results/freq_results', recursive = TRUE, showWarnings = FALSE)
dir.create('../../../data/rna/pseudobulk/results/freq_plots', recursive = TRUE, showWarnings = FALSE)

## 1. PBMC - Treatment

In [7]:
df <- fread('../../../data/rna/pseudobulk/outputs/pbmc_l3_frequency.csv')
df$tissue <- 'PBMC'

df <- df[df$label.visitDetails != '',]
df <- df[!is.na(label.visitDetails), ]

In [8]:
pbmc_tx_tp <- c(
    'PreTx',
    'PI2C',
    'EI',
    'ASCT60d',
    'ASCT1y',
    'ASCT2y',
    'Healthy'
)
df[, label.visitDetails := factor(label.visitDetails, levels = pbmc_tx_tp)]

In [9]:
visit_levels <- levels(df$label.visitDetails)
visit_levels

[1] "PreTx"   "PI2C"    "EI"      "ASCT60d" "ASCT1y"  "ASCT2y"  "Healthy"

### 1.1 Treatment Analysis - Paired Wilcoxon Test

In [10]:
adjacent_pairs <- data.frame(
  group1 = visit_levels[-length(visit_levels)],
  group2 = visit_levels[-1]
)
adjacent_pairs <- subset(adjacent_pairs, group2 != 'Healthy')

add_pairs <- data.frame(
  group1 = c('PreTx', 
             'EI', 
             'EI'),
  group2 = c('EI', 
             'ASCT1y', 
             'ASCT2y'),
  stringsAsFactors = FALSE
)

In [11]:
adjacent_pairs <- rbind(adjacent_pairs, add_pairs)
adjacent_pairs

group1,group2
<chr>,<chr>
PreTx,PI2C
PI2C,EI
EI,ASCT60d
ASCT60d,ASCT1y
ASCT1y,ASCT2y
PreTx,EI
EI,ASCT1y
EI,ASCT2y


In [12]:
pairs <- adjacent_pairs %>% transmute(g1 = group1, g2 = group2)

results <- pmap(pairs, ~{
  deg_res <- test_wilcox_paired(df, g1 = ..1, g2 = ..2,
                            freq_data = 'cell_type_clr',
                            label_col = 'aifi_plot_l3',
                            analysis_label = 'Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment)')
  if (nrow(deg_res)) plot_freq_volcano(df, deg_res,
                                   freq_data = 'cell_type_clr',
                                   paired = TRUE, col_g2 = 'blue4',
                                   effect_size = 0.05)
  deg_res
})

tx_vs_tx_results <- bind_rows(results)
tx_vs_tx_results$analysis_tag <- 'tx_vs_tx_timepoints'

Warning message:
“Use of .data in tidyselect expressions was deprecated in tidyselect 1.2.0.
ℹ Please use `all_of(var)` (or `any_of(var)`) instead of `.data[[var]]`”


### 1.2 Treatment vs Healthy Analysis - Unpaired Wilcoxon Test

In [13]:
healthy_pairs <- data.frame(
  group1 = setdiff(visit_levels, 'Healthy'),
  group2 = 'Healthy'
)

healthy_pairs

group1,group2
<chr>,<chr>
PreTx,Healthy
PI2C,Healthy
EI,Healthy
ASCT60d,Healthy
ASCT1y,Healthy
ASCT2y,Healthy


In [14]:
pairs <- healthy_pairs %>% transmute(g1 = group1, g2 = group2)

results <- pmap(pairs, ~{
  deg_res <- test_wilcox_unpaired(df, g1 = ..1, g2 = ..2,
                            freq_data = 'cell_type_clr',
                            label_col = 'aifi_plot_l3',
                            analysis_label = 'Timepoint Analysis (Unpaired Wilcoxon with sample level FDR adjustment)')
  if (nrow(deg_res)) plot_freq_volcano(df, 
                                   deg_res,
                                   freq_data = 'cell_type_clr',
                                   paired = FALSE, 
                                   col_g2 = 'green4',
                                   effect_size = 0.05)
  deg_res
})

tx_vs_healthy_results <- bind_rows(results)
tx_vs_tx_results$analysis_tag <- 'tx_vs_healthy_timepoints'

### 1.3 Combine tables into one and save results

In [15]:
combined_results <- bind_rows(tx_vs_tx_results, tx_vs_healthy_results)
combined_results$tissue <- 'PBMC'
write.csv(combined_results, '../../../data/rna/pseudobulk/results/freq_results/pbmc_all_timepoint_frequency_results.csv')

## 2. BMMC

In [16]:
df <- fread('../../../data/rna/pseudobulk/outputs/bmmc_l3_frequency.csv')
df$tissue <- 'BMMC'

df <- df[df$label.visitDetails != '',]
df <- df[!is.na(label.visitDetails), ]

In [17]:
head(df)

aifi_plot_l3,sample.sampleKitGuid,counts,total_counts,raw_frequency,pseudo_counts,pseudo_total_counts,pseudo_frequency,cell_type_clr,level_id,⋯,subject.birthYear,subject.ethnicity,subject.race,subject.subjectGuid,subject.cmv,cohort.cohortGuid,manual.time_stamp,manual.category,manual.flu_response,tissue
<chr>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<IDate>,<chr>,<chr>,<chr>
ASDC,KT01671,3,3260,0.0009202454,4,3322,0.0012040939,-1.5309212,l3,⋯,1971,None,Caucasian,CE0006259,,StemCell Technologies,2024-06-12,healthy_bmmc,None,BMMC
ASDC,KT02419,7,20804,0.0003364738,8,20866,0.0003833988,-2.2129544,l3,⋯,1956,Non-Hispanic origin,Caucasian,FH1001,,FH1,2024-02-24,tumor_bmmc,None,BMMC
ASDC,KT02420,5,10217,0.0004893804,6,10279,0.0005837144,-2.3908112,l3,⋯,1957,Non-Hispanic origin,Caucasian,FH1002,Positive,FH1,2024-02-24,tumor_bmmc,flu_responder,BMMC
ASDC,KT02421,3,10000,0.0003000000,4,10062,0.0003975353,-2.3697533,l3,⋯,1958,Non-Hispanic origin,Caucasian,FH1003,Positive,FH1,2023-10-24,tumor_bmmc,flu_non_responder,BMMC
ASDC,KT02422,13,12158,0.0010692548,14,12220,0.0011456628,-0.8881177,l3,⋯,1958,Non-Hispanic origin,Caucasian,FH1003,Positive,FH1,2023-10-24,tumor_bmmc,flu_non_responder,BMMC
ASDC,KT02423,11,5498,0.0020007275,12,5560,0.0021582734,-0.8527624,l3,⋯,1958,Non-Hispanic origin,Caucasian,FH1003,Positive,FH1,2023-10-24,tumor_bmmc,flu_non_responder,BMMC


In [18]:
bmmc_tx_tp <- c(
    'PreTx',
    'EI',
    'ASCT90d',
    'ASCT1y',
    'ASCT2y',
    'Healthy'
)
df[, label.visitDetails := factor(label.visitDetails, levels = bmmc_tx_tp)]

In [19]:
visit_levels <- levels(df$label.visitDetails)
visit_levels

[1] "PreTx"   "EI"      "ASCT90d" "ASCT1y"  "ASCT2y"  "Healthy"

In [20]:
table(df$label.visitDetails)


  PreTx      EI ASCT90d  ASCT1y  ASCT2y Healthy 
    992     744     806     682     434     620 

### 2.1 BMMC Treatment Analysis - Paired Wilcoxon Test

In [21]:
adjacent_pairs <- data.frame(
  group1 = visit_levels[-length(visit_levels)],
  group2 = visit_levels[-1]
)
adjacent_pairs <- subset(adjacent_pairs, group2 != 'Healthy')

add_pairs <- data.frame(
  group1 = c('EI', 
             'EI'),
  group2 = c('ASCT1y', 
             'ASCT2y'),
  stringsAsFactors = FALSE
)

In [22]:
adjacent_pairs <- rbind(adjacent_pairs, add_pairs)
adjacent_pairs

group1,group2
<chr>,<chr>
PreTx,EI
EI,ASCT90d
ASCT90d,ASCT1y
ASCT1y,ASCT2y
EI,ASCT1y
EI,ASCT2y


In [23]:
pairs <- adjacent_pairs %>% transmute(g1 = group1, g2 = group2)

results <- pmap(pairs, ~{
  deg_res <- test_wilcox_paired(df, g1 = ..1, g2 = ..2,
                            freq_data = 'cell_type_clr',
                            label_col = 'aifi_plot_l3',
                            analysis_label = 'Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment)')
  if (nrow(deg_res)) plot_freq_volcano(df, deg_res,
                                   freq_data = 'cell_type_clr',
                                   paired = TRUE, col_g2 = 'blue4',
                                   effect_size = 0.05)
  deg_res
})

tx_vs_tx_results <- bind_rows(results)
tx_vs_tx_results$analysis_tag <- 'tx_vs_tx_timepoints'

### 2.2 BMMC Treatment vs Healthy Analysis - Unpaired Wilcoxon Test

In [24]:
healthy_pairs <- data.frame(
  group1 = setdiff(visit_levels, 'Healthy'),
  group2 = 'Healthy'
)

healthy_pairs

group1,group2
<chr>,<chr>
PreTx,Healthy
EI,Healthy
ASCT90d,Healthy
ASCT1y,Healthy
ASCT2y,Healthy


In [26]:
pairs <- healthy_pairs %>% transmute(g1 = group1, g2 = group2)

results <- pmap(pairs, ~{
  deg_res <- test_wilcox_unpaired(df, g1 = ..1, g2 = ..2,
                            freq_data = 'cell_type_clr',
                            label_col = 'aifi_plot_l3',
                            analysis_label = 'Timepoint Analysis (Unpaired Wilcoxon with sample level FDR adjustment)')
  if (nrow(deg_res)) plot_freq_volcano(df, 
                                   deg_res,
                                   freq_data = 'cell_type_clr',
                                   paired = FALSE, 
                                   col_g2 = 'green4',
                                   effect_size = 0.05)
  deg_res
})

tx_vs_healthy_results <- bind_rows(results)
tx_vs_healthy_results$analysis_tag <- 'tx_vs_healthy_timepoints'

### 2.3 Combine tables into one and save results

In [27]:
combined_results <- bind_rows(tx_vs_tx_results, tx_vs_healthy_results)
combined_results$tissue <- 'BMMC'
write.csv(combined_results, '../../../data/rna/pseudobulk/results/freq_results/bmmc_all_timepoint_frequency_results.csv')

In [28]:
head(combined_results)

group1,group2,analysis_label,aifi_plot_l3,n,statistic,p,p.adj,analysis_tag,n1,n2,tissue
<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<chr>
PreTx,EI,Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment),ASDC,11,0,0.0009765625,0.02018229,tx_vs_tx_timepoints,NA,NA,BMMC
PreTx,EI,Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment),BaEoMaP,11,23,0.4130859375,0.51222656,tx_vs_tx_timepoints,NA,NA,BMMC
PreTx,EI,Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment),CD14 Mono Core,11,7,0.0185546875,0.10458097,tx_vs_tx_timepoints,NA,NA,BMMC
PreTx,EI,Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment),CD14 Mono ISG+,11,23,0.4130859375,0.51222656,tx_vs_tx_timepoints,NA,NA,BMMC
PreTx,EI,Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment),CD16 Mono Core,11,20,0.2783203125,0.42087462,tx_vs_tx_timepoints,NA,NA,BMMC
PreTx,EI,Timepoint Analysis (Paired Wilcoxon with sample level FDR adjustment),CD4 T CM,11,18,0.2060546875,0.33619449,tx_vs_tx_timepoints,NA,NA,BMMC
